> ## 2026 갱신 안내 (최신 Colab 대응)
>
> 이 노트북은 약 1년 전 작성되어 최신 라이브러리에서 실행이 깨졌던 부분을 2026년 기준 현행 API로 갱신한 버전입니다. 위에서 아래로 순차 실행하면 동작하도록 정리했습니다.
>
> **주요 변경 사항**
> - **모델 클래스 수정**: `zjunlp/MolGen-large` 는 실제로 **BART 계열 seq2seq(`BartForConditionalGeneration`)** 모델입니다(HF `config.json` 의 `is_encoder_decoder: true`, `architectures: [BartForConditionalGeneration]` 확인). 따라서 `AutoModelForCausalLM` 은 부적절하며 **`AutoModelForSeq2SeqLM`** 로 교체했습니다(로딩 실패 시 `trust_remote_code=True` 재시도).
> - **생성 API 현대화**: `model.generate(...)` 에서 deprecated 된 `max_length` 대신 `max_new_tokens` 를 사용하고 `do_sample / top_k / top_p / num_return_sequences` 를 명시했습니다. 생성된 SELFIES 를 SMILES 로 변환할 때 invalid/None 을 방어합니다.
> - **설치 셀**: 구버전 핀을 제거하고 `!pip install -q -U transformers datasets selfies rdkit accelerate` 로 최신본을 설치합니다.
> - **파인튜닝 셀**: `evaluation_strategy -> eval_strategy`, Trainer `tokenizer= -> processing_class=`, `DataCollatorForSeq2Seq` 사용으로 현행 `transformers` 에 맞췄습니다. 파인튜닝은 GPU/시간이 필요하므로 **소규모·짧게 도는 옵션(`RUN_FINETUNE`)** 으로 만들었고, 데이터 URL 이 죽었을 경우 소규모 대체 데이터로 폴백합니다.
>
> **남은 리스크 (Colab 실제 실행 시 확인 필요)**
> - `zjunlp/MolGen-large` 는 약 1.4GB 다운로드가 필요합니다(HF 접속/네트워크 필요). CPU 로도 로딩·생성은 되지만 느립니다.
> - 파인튜닝은 기본적으로 꺼져 있습니다(`RUN_FINETUNE = False`). 켤 경우 GPU 런타임을 권장합니다.
> - `plogp_test.csv` 원본 URL 이 바뀌거나 죽으면 자동으로 소규모 대체 데이터를 씁니다.

<a href="https://colab.research.google.com/github/fourmodern/toc_tutorial_colab/blob/main/teachopencadd/t041_molgen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MolGen Tutorial**

MolGen은 다양한 분자 생성 및 변환 작업을 수행하기 위한 통합 프레임워크로, 사전 학습된 분자 생성 모델들을 기반으로 분자 디자인, 약물 발견, 화합물 최적화 등을 지원합니다. 이 프레임워크는 최신 딥러닝 아키텍처를 활용하며, 분자 표현으로는 SMILES 및 SELFIES를 지원하여 다양한 분자 표현 작업을 수행할 수 있습니다.

### **지원되는 주요 기능**

MolGen은 여러 분자 생성 및 변환 작업을 수행할 수 있도록 다양한 모델을 통합하고 있으며, 다음과 같은 특징을 가지고 있습니다:
- **다양한 입력 포맷 지원**: SMILES, SELFIES 등의 분자 표현을 지원하여 분자 데이터를 효과적으로 처리할 수 있습니다.
- **Fine-Tuning 지원**: MolGen은 다양한 데이터셋으로 모델을 fine-tuning하여 특정 분자 생성 및 변환 작업에 최적화할 수 있습니다.
- **Deep Learning 통합**: 최신 딥러닝 모델을 활용하여 분자 생성 작업을 수행하며, DeepSpeed와 Hugging Face `transformers` 라이브러리와의 통합을 지원합니다.

### **Citation**

MolGen을 사용하거나 참고하는 연구를 진행할 경우, 다음과 같이 인용할 수 있습니다:

```bibtex
@article{fang2022molt5,
  title={MolGen: A Pre-trained Generative Model for Molecular Generation and Optimization},
  author={Fang, Xiaomin and Wu, Hongyi and Zeng, Xiantao and Chen, Junqi and Song, Jiawei and Yang, Huanqin and Shen, Yuhu},
  journal={arXiv preprint arXiv:2204.11817},
  year={2022}
}

## 0. **Prerequisites**

In [ ]:
# 최신 버전으로 설치 (구버전 핀 제거). accelerate 는 Trainer/large 모델 로딩에 도움.
!pip install -q -U transformers datasets selfies rdkit accelerate
# torch 는 Colab 에 이미 설치되어 있어 재설치하지 않습니다 (런타임 CUDA 빌드와 어긋날 수 있음)
# !pip install -q torch

## **1.Molecule Generation (Before Fine-Tuning)**

### 1.1 Import Required Libraries and Load MolGen Model

Hugging Face에서 MolGen 모델을 불러옵니다.


In [ ]:
# MolGen-large 는 BART 계열 seq2seq 모델(BartForConditionalGeneration)이므로
# AutoModelForCausalLM 이 아니라 AutoModelForSeq2SeqLM 을 사용해야 합니다.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import selfies as sf
import torch

model_name = "zjunlp/MolGen-large"

# Hugging Face 에서 MolGen 토크나이저 / 모델 로드
tokenizer = AutoTokenizer.from_pretrained(model_name)
try:
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
except Exception as e:
    print("기본 로딩 실패, trust_remote_code=True 로 재시도합니다:", e)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name, trust_remote_code=True)

# GPU 가 있으면 GPU 로 이동 (CPU 로도 동작하지만 느립니다)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()
print(f"모델 로드 완료: {model_name} ({model.__class__.__name__}) on {device}")

### 1.2 Converting SMILES to SELFIES and Generating Molecules

SMILES를 SELFIES로 변환한 후 모델을 통해 분자를 생성합니다.

In [ ]:
# Example SMILES input
smiles_input = "CCO"  # Ethanol as an example

# Convert SMILES to SELFIES (selfies 2.x)
selfies_input = sf.encoder(smiles_input)
print("Input SELFIES:", selfies_input)

# SELFIES 를 SMILES 로 안전하게 변환 (invalid/None 방어)
def safe_selfies_to_smiles(selfies_str):
    if not selfies_str:
        return None
    # 토크나이저 디코딩 결과에 공백이 섞일 수 있으므로 제거
    cleaned = selfies_str.replace(" ", "")
    try:
        smi = sf.decoder(cleaned)
    except Exception:
        return None
    return smi if smi else None


# seq2seq 모델로 분자를 생성하는 함수
def generate_molecules_from_selfies(selfies_input, num_sequences=5, max_new_tokens=100):
    # 입력 SELFIES 를 토큰화
    input_ids = tokenizer(selfies_input, return_tensors="pt").input_ids.to(model.device)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,          # deprecated 된 max_length 대신 사용
            num_return_sequences=num_sequences,
            do_sample=True,
            top_k=50,
            top_p=0.95,
        )

    # 생성된 SELFIES 를 디코딩 후 SMILES 로 변환 (None 방어)
    generated_selfies = tokenizer.batch_decode(output, skip_special_tokens=True)
    generated_smiles = [safe_selfies_to_smiles(s) for s in generated_selfies]
    return generated_smiles


# Fine-tuning 전 분자 생성
molecules_before = generate_molecules_from_selfies(selfies_input)
print("Generated Molecules (Before Fine-tuning):")
for idx, mol in enumerate(molecules_before):
    print(f"Molecule {idx + 1}: {mol}")

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display

# 입력된 molecule과 생성된 molecule을 한 줄에 그려주는 함수
def visualize_molecules_in_grid(input_smiles, generated_smiles_list):
    # 입력된 SMILES를 RDKit Mol 객체로 변환
    input_mol = Chem.MolFromSmiles(input_smiles)

    # 생성된 SMILES들을 RDKit Mol 객체로 변환 (None/invalid 방어)
    generated_mols = []
    for smiles in generated_smiles_list:
        if not smiles:
            continue
        m = Chem.MolFromSmiles(smiles)
        if m is not None:
            generated_mols.append(m)

    # 입력된 molecule을 맨 앞에 추가
    all_mols = ([input_mol] if input_mol is not None else []) + generated_mols
    if not all_mols:
        print("시각화할 유효한 분자가 없습니다.")
        return

    # 각 molecule의 레이블
    legends = (["Input"] if input_mol is not None else []) + [
        f"Generated {i + 1}" for i in range(len(generated_mols))
    ]

    # 한 줄에 분자들을 그리는 grid image 생성
    img = Draw.MolsToGridImage(
        all_mols, molsPerRow=len(all_mols), subImgSize=(300, 300), legends=legends
    )
    display(img)


# Visualization 실행
visualize_molecules_in_grid(smiles_input, molecules_before)

## **2. Fine-Tuning the Model**

### 2.1 Preparing Dataset for Fine-Tuning

사용자 데이터셋을 SELFIES로 변환하여 fine-tuning에 활용합니다.

In [ ]:
import os
import urllib.request
import pandas as pd

# plogp_test.csv 다운로드 (URL 이 죽었을 경우 소규모 대체 데이터로 폴백)
file_url = "https://raw.githubusercontent.com/zjunlp/MolGen/main/moldata/finetune/plogp_test.csv"
csv_file = "plogp_test.csv"

have_data = False
try:
    urllib.request.urlretrieve(file_url, csv_file)
    have_data = os.path.exists(csv_file) and os.path.getsize(csv_file) > 0
    if have_data:
        print(f"데이터 다운로드 완료: {csv_file} ({os.path.getsize(csv_file)} bytes)")
except Exception as e:
    print("다운로드 실패, 소규모 대체 데이터를 생성합니다:", e)

if not have_data:
    # 파인튜닝 데모용 소규모 대체 SMILES (실데이터 다운로드 실패 시에만 사용)
    fallback_smiles = [
        "CCO", "CCN", "CCC", "CCCC", "c1ccccc1", "CC(=O)O", "CCOC(=O)C",
        "CN1CCCC1", "c1ccncc1", "OCC(O)CO", "CC(C)O", "CCOCC",
        "c1ccc(O)cc1", "CC(=O)Nc1ccccc1", "COc1ccccc1", "CCCCO",
        "CC(=O)OC", "NCCO", "c1ccc(N)cc1", "CC(C)(C)O",
    ]
    pd.DataFrame({"smiles": fallback_smiles}).to_csv(csv_file, index=False)
    print(f"소규모 대체 데이터 생성 완료: {csv_file} ({len(fallback_smiles)} molecules)")

In [ ]:
import torch
import selfies as sf
from datasets import load_dataset

# 파인튜닝은 시간이 걸리므로 소규모 데모용 샘플 수만 사용 (전체를 쓰려면 None 으로)
MAX_TRAIN_SAMPLES = 200
MAX_TOKEN_LEN = 128  # SELFIES 시퀀스 최대 토큰 길이 (패딩/절단 기준)

# CSV 파일 로드
dataset = load_dataset("csv", data_files=csv_file)

# 소규모 subset 선택 (데모용)
if MAX_TRAIN_SAMPLES is not None:
    n = min(MAX_TRAIN_SAMPLES, len(dataset["train"]))
    dataset["train"] = dataset["train"].select(range(n))

print("데이터셋 예시:", dataset["train"][0])


# SMILES를 SELFIES로 변환 (invalid SMILES 방어)
def smiles_to_selfies(example):
    try:
        example["selfies"] = sf.encoder(example["smiles"])
    except Exception:
        example["selfies"] = None
    return example


dataset = dataset.map(smiles_to_selfies)
# 변환 실패한 행 제거
dataset = dataset.filter(lambda ex: ex["selfies"] is not None)
print("SELFIES 변환 후 학습 샘플 수:", len(dataset["train"]))


# SELFIES를 토큰화하고 'labels'를 추가 (seq2seq: labels = input_ids)
def tokenize_selfies_function(examples):
    tokenized = tokenizer(
        examples["selfies"],
        padding="max_length",
        truncation=True,
        max_length=MAX_TOKEN_LEN,
    )
    tokenized["labels"] = [ids.copy() for ids in tokenized["input_ids"]]
    return tokenized


tokenized_dataset = dataset.map(tokenize_selfies_function, batched=True)

### 2.2 Performing Fine-Tuning

In [ ]:
from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
)

# 파인튜닝은 GPU/시간이 필요하므로 기본은 비활성화. True 로 바꾸면 소규모로 짧게 학습합니다.
RUN_FINETUNE = False

if RUN_FINETUNE:
    use_fp16 = torch.cuda.is_available()  # fp16 은 GPU 에서만 (CPU 에서 fp16=True 는 에러)

    # 동적 패딩 collator (seq2seq)
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

    # TrainingArguments (현행 transformers: evaluation_strategy -> eval_strategy)
    training_args = TrainingArguments(
        output_dir="./results",
        overwrite_output_dir=True,
        num_train_epochs=1,                # 데모용으로 짧게
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        save_steps=50,
        save_total_limit=1,
        fp16=use_fp16,
        learning_rate=1e-4,
        weight_decay=1e-4,
        eval_strategy="no",                # 데모 단순화 (평가 생략)
        logging_dir="./logs",
        logging_steps=10,
        report_to="none",
    )

    # Trainer (현행 transformers: tokenizer= -> processing_class=)
    try:
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tokenized_dataset["train"],
            data_collator=data_collator,
            processing_class=tokenizer,    # 신버전 인자명
        )
    except TypeError:
        # 매우 구버전 호환 (processing_class 미지원 시)
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tokenized_dataset["train"],
            data_collator=data_collator,
            tokenizer=tokenizer,
        )

    trainer.train()
    print("파인튜닝 완료.")
else:
    print(
        "RUN_FINETUNE = False 이므로 파인튜닝을 건너뜁니다.\n"
        "파인튜닝을 실행하려면 위의 RUN_FINETUNE 을 True 로 바꾸세요 (GPU 런타임 권장)."
    )

## **3. After Fine-Tuning Generation and Visualization**

### 3.1 Generating Molecules After Fine-Tuning

Fine-tuning이 완료된 모델을 사용하여 새로운 분자를 생성합니다.

In [ ]:
# 파인튜닝 이후(또는 RUN_FINETUNE=False 인 경우 파인튜닝 없이) 분자 생성
# 위 1.2 절에서 정의한 generate_molecules_from_selfies / safe_selfies_to_smiles 를 재사용합니다.
model.eval()

molecules_after = generate_molecules_from_selfies(selfies_input)
label = "After Fine-tuning" if RUN_FINETUNE else "After (파인튜닝 미실행 — 참고용)"
print(f"Generated Molecules ({label}):")
for idx, mol in enumerate(molecules_after):
    print(f"Molecule {idx + 1}: {mol}")

### 3.2 Visualizing Changes Using RDKit

RDKit를 사용하여 fine-tuning 전후의 분자를 시각화합니다.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display


# Convert SMILES to RDKit Mol object (None/invalid 방어)
def smiles_to_mol(smiles):
    if not smiles:
        return None
    return Chem.MolFromSmiles(smiles)


# Function to visualize molecule changes
def visualize_molecule_changes(smiles_before, smiles_after):
    mol_before = smiles_to_mol(smiles_before)
    mol_after = smiles_to_mol(smiles_after)

    if mol_before is not None and mol_after is not None:
        img_before = Draw.MolToImage(mol_before, size=(300, 300), legend="Before")
        img_after = Draw.MolToImage(mol_after, size=(300, 300), legend="After")
        print("Before Fine-tuning:")
        display(img_before)
        print("After Fine-tuning:")
        display(img_after)
    else:
        print("Invalid SMILES strings detected. Unable to generate molecule visualization.")


# Visualize changes for each generated molecule
def visualize_changes(molecules_before, molecules_after):
    for idx in range(min(len(molecules_before), len(molecules_after))):
        print(f"Molecule {idx + 1}:")
        print(f"SMILES Before: {molecules_before[idx]}")
        print(f"SMILES After: {molecules_after[idx]}")
        visualize_molecule_changes(molecules_before[idx], molecules_after[idx])
        print("\n")


# Visualize the changes between pre- and post-fine-tuning molecules
visualize_changes(molecules_before, molecules_after)

# **튜토리얼 요약**
1. Molecule Generation (Before Fine-Tuning): SMILES 입력을 SELFIES로 변환하여 모델에 전달하고, 생성된 SELFIES를 다시 SMILES로 변환하여 확인합니다.
2. Fine-Tuning: 데이터셋의 SMILES를 SELFIES로 변환하여 모델을 fine-tuning합니다.
3. After Fine-Tuning Generation 및 Visualization: Fine-tuning 이후 생성된 결과를 시각화하고 구조 변화를 확인합니다.

---

## References

- Fang, Xiaomin, et al. "MolGen: A Pre-trained Generative Model for Molecular Generation and Optimization." *arXiv preprint arXiv:2204.11817*, 2022.
  
  Available at: [MolGen Paper on arXiv](https://arxiv.org/abs/2204.11817)

- MolGen GitHub Repository: [https://github.com/zjunlp/MolGen](https://github.com/zjunlp/MolGen)

- RDKit: Open-source cheminformatics software. Available at: [https://www.rdkit.org](https://www.rdkit.org)

- SELFIES: Krenn, Mario, et al. "SELFIES: a robust representation of semantically constrained graphs with an example application in chemistry." *Machine Learning: Science and Technology*, 2020.
  
  Available at: [https://github.com/aspuru-guzik-group/selfies](https://github.com/aspuru-guzik-group/selfies)

- Hugging Face Transformers: Wolf, Thomas, et al. "Transformers: State-of-the-Art Natural Language Processing." *arXiv preprint arXiv:1910.03771*, 2019.

  Available at: [https://huggingface.co/transformers](https://huggingface.co/transformers)